In [2]:
import torch

from lucent.optvis import render, param, transform
from lucent_layer_utils import get_visualizable_layers
from lucent.modelzoo import inceptionv1, inception_v3, resnet152, resnext101_64x4d

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

model = inceptionv1(pretrained=True)
# model = inception_v3(pretrained=True)
# model = resnet152(pretrained=True)
# model = resnext101_64x4d(pretrained=True)
_ = model.to(device).eval()


good_layers = get_visualizable_layers(model)
print(good_layers)

cuda:0
['conv2d0', 'conv2d1', 'conv2d2', 'mixed3a_1x1', 'mixed3a_3x3_bottleneck', 'mixed3a_5x5_bottleneck', 'mixed3a_3x3', 'mixed3a_5x5', 'mixed3a', 'mixed3b_1x1', 'mixed3b_3x3_bottleneck', 'mixed3b_5x5_bottleneck', 'mixed3b_3x3', 'mixed3b_5x5', 'mixed3b', 'mixed4a_1x1', 'mixed4a_3x3_bottleneck', 'mixed4a_5x5_bottleneck', 'mixed4a_3x3', 'mixed4a_5x5', 'mixed4a', 'mixed4b_1x1', 'mixed4b_3x3_bottleneck', 'mixed4b_5x5_bottleneck', 'mixed4b_3x3', 'mixed4b_5x5', 'mixed4b', 'mixed4c_1x1', 'mixed4c_3x3_bottleneck', 'mixed4c_5x5_bottleneck', 'mixed4c_3x3', 'mixed4c_5x5', 'mixed4c', 'mixed4d_1x1', 'mixed4d_3x3_bottleneck', 'mixed4d_5x5_bottleneck', 'mixed4d_3x3', 'mixed4d_5x5', 'mixed4d', 'mixed4e_1x1', 'mixed4e_3x3_bottleneck', 'mixed4e_5x5_bottleneck', 'mixed4e_3x3', 'mixed4e_5x5', 'mixed4e', 'mixed5a_1x1', 'mixed5a_3x3_bottleneck', 'mixed5a_5x5_bottleneck', 'mixed5a_3x3', 'mixed5a_5x5', 'mixed5a', 'mixed5b_1x1', 'mixed5b_3x3_bottleneck', 'mixed5b_5x5_bottleneck', 'mixed5b_3x3', 'mixed5b_5x5'

In [3]:
from parse_model_lucent import create_random_objective


obj = create_random_objective(model, good_layers)  # Use first 30 for reliability
_ = render.render_vis(model, obj, param_f=lambda: param.image(128), show_image=True)

Selected layer 0: mixed3a_5x5_bottleneck
Available channels: 0-15
Selected channel 0: 4
Selected layer 1: mixed3a_5x5_bottleneck
Available channels: 0-15
Selected channel 1: 10


100%|██████████| 512/512 [00:08<00:00, 58.73it/s]


In [ ]:
other_obj = create_random_objective(model, good_layers)

combined_obj = obj + other_obj  # Combining two channels
_ = render.render_vis(
    model, combined_obj, param_f=lambda: param.image(128), show_image=True
)

In [ ]:
# hoping for a favorite!
obj = create_random_objective(model, good_layers, objective_type="center_3x3")
_ = render.render_vis(
    model, objective_f=obj, param_f=lambda: param.image(128), show_image=True
)

Selected layer 0: mixed4a_1x1
Available channels: 0-191
Selected channel 0: 180
Selected layer 1: mixed4a_1x1
Available channels: 0-191
Selected channel 1: 163


100%|██████████| 512/512 [00:11<00:00, 45.32it/s]


In [ ]:
# A visualization with gradient descent in pixel space
# Notice the high frequency components similar to adversarial images


def param_f():
    return param.image(128, fft=False, decorrelate=False)


# We set transforms=[] to denote no transforms
_ = render.render_vis(model, obj, param_f, transforms=[], show_inline=True)

In [ ]:
# A visualization with gradient descent in Fourier basis


def param_f():
    return param.image(128, fft=True, decorrelate=False)


_ = render.render_vis(model, obj, param_f, transforms=[], show_inline=True)

In [ ]:
# A visualization with channel decorrelation


def param_f():
    return param.image(128, fft=False, decorrelate=True)


_ = render.render_vis(model, obj, param_f, transforms=[], show_inline=True)

In [ ]:
# A visualization with gradient descent in Fourier basis + channel decorrelation
# This is the default parameterization without transforms (see below)


def param_f():
    return param.image(128, fft=True, decorrelate=True)


_ = render.render_vis(model, obj, param_f, transforms=[], show_inline=True)

In [ ]:
# A visualization with CPPN parameterization


def cppn_param_f():
    return param.cppn(128)


# We initialize an optimizer with lower learning rate for CPPN
def cppn_opt(params):
    return torch.optim.Adam(params, 5e-3)


_ = render.render_vis(
    model, obj, cppn_param_f, cppn_opt, transforms=[], show_inline=True
)

In [ ]:
# No transformations, similar to our example earlier
_ = render.render_vis(model, obj, transforms=[], show_inline=True)

In [ ]:
# Adding jitter, notice that the visualization is much less noisy!

jitter_only = [transform.jitter(8)]

_ = render.render_vis(
    model, 
    obj, 
    transforms=jitter_only, 
    show_image=True, # show_inline=True,
)

In [ ]:
# Adding a whole suite of transforms!

all_transforms = [
    transform.pad(16),
    transform.jitter(8),
    transform.random_scale([n / 100.0 for n in range(80, 120)]),
    transform.random_rotate(
        list(range(-10, 10)) + list(range(-5, 5)) + 10 * list(range(-2, 2))
    ),
    transform.jitter(2),
]

_ = render.render_vis(
    model, 
    objective_f=obj, 
    param_f=lambda: param.image(128),
    transforms=all_transforms,     
    show_image=True, # show_inline=True,
)

In [ ]:
# Adding jitter with CPPN

_ = render.render_vis(
    model,
    combined_obj,
    cppn_param_f,
    cppn_opt,
    transforms=jitter_only,
    show_inline=True,
)

In [ ]:
# Adding a suite of transforms with CPPN

_ = render.render_vis(
    model,
    combined_obj,
    cppn_param_f,
    cppn_opt,
    transforms=all_transforms,
    show_inline=True,
)